# Treino — classificador de cor/raça autodeclarada (TSE)

1. **Ambiente de execução → Alterar o tipo de ambiente → GPU T4**.
2. O zip precisa estar no Drive em `Meu Drive/pardos/pardos_tse_coloridas.zip`.
3. Rode as células em ordem. Modelo e métricas vão para `Meu Drive/pardos/resultados/` (sobrevivem se o Colab desconectar).

Dados sensíveis (rosto + cor/raça): não compartilhe o notebook com saídas nem a pasta do Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "timm>=1.0" 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ZIP = '/content/drive/MyDrive/pardos/pardos_tse_coloridas.zip'
SAIDA = '/content/drive/MyDrive/pardos/resultados'
assert os.path.exists(ZIP), 'zip não encontrado no Drive: ' + ZIP
os.makedirs(SAIDA, exist_ok=True)
# copia para o disco local da VM: ler 125 mil jpgs direto do Drive é muito lento
!cp "$ZIP" /content/dados.zip && mkdir -p /content/dados && unzip -q -o /content/dados.zip -d /content/dados && rm /content/dados.zip
!ls /content/dados && du -sh /content/dados

In [ ]:
%%writefile /content/treino.py
"""Treino e avaliação do classificador de cor/raça autodeclarada (5 classes do IBGE).

Dois modos:
  --modo sonda     extrai embeddings de um backbone congelado (CLIP ViT-B/16) e treina
                   regressão logística. Linha de base barata; roda até em CPU com subamostra.
  --modo ajuste    fine-tuning do backbone inteiro (precisa de GPU: Kaggle).

Treina SÓ com pessoas de declaração consistente (n_racas == 1). Avalia em:
  - test consistente: matriz de confusão, recall por classe, acurácia balanceada,
    e a métrica binária "parda vs. resto"
  - conjunto ambíguo (pessoas que mudaram de declaração): para onde o modelo as joga,
    comparado com o histórico de declarações delas

Uso local (teste rápido):  python treino.py --modo sonda --por-classe 300
GPU (Colab):               python treino.py --modo ajuste --dados /content/dados --epocas 8
Indígena vs. resto:        python treino.py --modo ajuste --binario indigena ...
"""
import argparse
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset

CLASSES = ["branca", "parda", "preta", "indigena", "amarela"]  # redefinida por --classes
BACKBONE = "vit_base_patch16_clip_224.openai"                  # redefinido por --backbone
RECORTE = 1.0  # fração central do recorte 160x160 usada (--recorte-central); 1 = inteiro
ORIGEM = CLASSES  # raças do TSE que entram (--classes); difere de CLASSES só no modo binário
BINARIO = None    # --binario indigena: CLASSES vira ["outra", "indigena"]
ALVO = "parda"    # classe da métrica "X vs. resto"
WORKERS = min(8, os.cpu_count() or 2)


def rotular(df: pd.DataFrame) -> pd.DataFrame:
    """No modo binário, troca o rótulo por alvo/"outra" (inclusive no histórico)."""
    if not BINARIO:
        return df
    mapa = lambda r: r if r == BINARIO else "outra"
    df = df.assign(raca=df.raca.map(mapa))
    if "racas_historico" in df:
        df = df.assign(racas_historico=df.racas_historico.map(
            lambda h: "|".join(sorted({mapa(r) for r in h.split("|")}))))
    return df


class Rostos(Dataset):
    def __init__(self, df, raiz, transform):
        self.arquivos = [raiz / a for a in df.arquivo]
        self.y = df.raca.map(CLASSES.index).to_numpy()
        self.t = transform

    def __len__(self):
        return len(self.arquivos)

    def __getitem__(self, i):
        img = Image.open(self.arquivos[i]).convert("RGB")
        if RECORTE < 1:
            # olhos ficam a 40% da altura: centraliza o corte um pouco abaixo deles
            w, h = img.size
            lado = RECORTE * w
            cx, cy = w / 2, h * 0.5
            img = img.crop((round(cx - lado / 2), round(cy - lado / 2),
                            round(cx + lado / 2), round(cy + lado / 2)))
        return self.t(img), self.y[i]


def carregar(raiz: Path, por_classe: int | None, semente: int):
    m = pd.read_csv(raiz / "manifesto.csv", dtype={"sq": str, "pessoa_id": str})
    m = m[(m.status == "ok") & m.raca.isin(ORIGEM)]
    if "saturacao" in m:  # 2016 inteiro e parte de 2018/2020 vieram em preto e branco
        m = m[m.saturacao >= 12]
    # ambíguo: só quem transitou entre classes que estão no modelo
    m = m[m.racas_historico.map(lambda h: all(r in ORIGEM for r in h.split("|")))]
    m = rotular(m)
    n_hist = m.racas_historico.str.count(r"\|") + 1
    cons = m[n_hist == 1]   # no binário, quem foi parda|branca é "outra" consistente
    ambiguo = m[n_hist > 1]
    partes = {s: cons[cons.split == s] for s in ["train", "val", "test"]}
    if por_classe:
        # limita por PESSOA (todas as fotos da pessoa vêm juntas)
        def limitar(df, n):
            pessoas = (df.drop_duplicates("pessoa_id").groupby("raca", group_keys=False)
                       .apply(lambda g: g.sample(min(len(g), n), random_state=semente)))
            return df[df.pessoa_id.isin(pessoas.pessoa_id)]
        partes["train"] = limitar(partes["train"], por_classe)
        for s in ["val", "test"]:
            partes[s] = limitar(partes[s], max(50, por_classe // 8))
        ambiguo = ambiguo.sample(min(len(ambiguo), por_classe * 2), random_state=semente)
    return partes, ambiguo


def relatorio(y, p_prob, titulo):
    pred = p_prob.argmax(1)
    presentes = sorted(set(y) | set(pred))
    print(f"\n=== {titulo} ({len(y)} fotos) ===")
    print(f"acurácia balanceada: {balanced_accuracy_score(y, pred):.3f}")
    print(classification_report(y, pred, labels=presentes,
                                target_names=[CLASSES[i] for i in presentes], digits=3,
                                zero_division=0))
    cm = pd.DataFrame(confusion_matrix(y, pred, labels=range(len(CLASSES))),
                      index=[f"real {c}" for c in CLASSES], columns=CLASSES)
    print(cm.to_string())
    ip = CLASSES.index(ALVO)
    yb, pb = (y == ip), (pred == ip)
    tp = (yb & pb).sum()
    print(f"\n{ALVO} vs. resto: precisão {tp / max(pb.sum(), 1):.3f}  "
          f"recall {tp / max(yb.sum(), 1):.3f}  acurácia {(yb == pb).mean():.3f}")
    return {"acuracia_balanceada": float(balanced_accuracy_score(y, pred)),
            "matriz": cm.values.tolist()}


def relatorio_ambiguo(amb: pd.DataFrame, prob: np.ndarray):
    amb = amb.assign(previsto=[CLASSES[i] for i in prob.argmax(1)])
    print(f"\n=== conjunto ambíguo ({len(amb)} fotos de quem mudou de declaração) ===")
    print("linhas = histórico de declarações; colunas = classe prevista (% da linha)")
    top = amb.racas_historico.value_counts().head(6).index
    t = pd.crosstab(amb[amb.racas_historico.isin(top)].racas_historico,
                    amb[amb.racas_historico.isin(top)].previsto, normalize="index")
    print((t * 100).round(1).to_string())
    acerto_hist = np.mean([p in h.split("|") for p, h in zip(amb.previsto, amb.racas_historico)])
    print(f"previsão bate com ALGUMA das declarações da pessoa: {acerto_hist:.1%}")


def relatorio_externo(ext: pd.DataFrame, prob: np.ndarray):
    """Fotos fora do padrão TSE. Cada fonte tem uma classe só, então o que dá para medir
    é: dentro de cada fonte, quanto o modelo acerta e para onde vão os erros."""
    ext = ext.assign(previsto=[CLASSES[i] for i in prob.argmax(1)],
                     p_alvo=prob[:, CLASSES.index(ALVO)])
    print(f"\n=== TESTE EXTERNO ({len(ext)} fotos fora do TSE) ===")
    print("linhas = fonte (rótulo); colunas = classe prevista (% da linha)")
    t = pd.crosstab(ext.fonte + " [" + ext.raca + "]", ext.previsto, normalize="index")
    print((t.reindex(columns=CLASSES, fill_value=0) * 100).round(1).to_string())
    pos = ext.raca == ALVO
    pred = ext.previsto == ALVO
    tp = (pos & pred).sum()
    print(f"\n{ALVO} vs. resto no externo: recall {tp / max(pos.sum(), 1):.3f}  "
          f"precisão {tp / max(pred.sum(), 1):.3f}  "
          f"(precisão aqui depende da mistura de fontes: leia com cuidado)")
    return {f: float((g.previsto == g.raca).mean()) for f, g in ext.groupby("fonte")}


def transformacoes(modelo, treino: bool):
    cfg = timm.data.resolve_data_config({}, model=modelo)
    return timm.data.create_transform(**cfg, is_training=treino,
                                      hflip=0.5 if treino else 0.0,
                                      scale=(0.6, 1.0), ratio=(0.9, 1.1),  # rosto já alinhado
                                      color_jitter=None)  # cor é o sinal: não distorcer


@torch.no_grad()
def embeddings(modelo, df, raiz, dev, lote):
    """Embeddings normalizados, com cache em disco por (backbone, recorte).

    Só calcula as imagens que ainda não estão no cache: trocar classes, amostra ou
    classificador não exige recalcular nada."""
    cache_path = raiz / "cache" / f"emb_{BACKBONE.replace('/', '_')}_rc{RECORTE}.npz"
    cache = {}
    if cache_path.exists():
        with np.load(cache_path) as z:
            cache = dict(zip(z["arquivos"], z["vetores"]))
    faltam = df[~df.arquivo.isin(cache)].drop_duplicates("arquivo")
    if len(faltam):
        print(f"  calculando {len(faltam)} embeddings ({len(df) - len(faltam)} no cache)",
              flush=True)
        dl = DataLoader(Rostos(faltam.assign(raca=CLASSES[0]), raiz,
                               transformacoes(modelo, False)), batch_size=lote, num_workers=WORKERS)
        novos = np.concatenate([modelo(x.to(dev)).float().cpu().numpy() for x, _ in dl])
        novos /= np.linalg.norm(novos, axis=1, keepdims=True)
        cache.update(zip(faltam.arquivo, novos))
        cache_path.parent.mkdir(exist_ok=True)
        tmp = cache_path.with_suffix(".tmp.npz")
        np.savez(tmp, arquivos=np.array(list(cache)), vetores=np.stack(list(cache.values())))
        tmp.rename(cache_path)
    return np.stack([cache[a] for a in df.arquivo])


def modo_sonda(partes, ambiguo, externo, raiz, dev, lote):
    modelo = timm.create_model(BACKBONE, pretrained=True, num_classes=0).to(dev).eval()
    t0 = time.time()
    emb = {s: embeddings(modelo, df, raiz, dev, lote) for s, df in partes.items()}
    emb_amb = embeddings(modelo, ambiguo, raiz, dev, lote)
    emb_ext = embeddings(modelo, externo, raiz, dev, lote) if externo is not None else None
    print(f"embeddings em {time.time() - t0:.0f}s")
    y = {s: df.raca.map(CLASSES.index).to_numpy() for s, df in partes.items()}
    melhor = None
    for C in [0.3, 1, 3, 10]:
        clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
        clf.fit(emb["train"], y["train"])
        ba = balanced_accuracy_score(y["val"], clf.predict(emb["val"]))
        print(f"C={C}: val acurácia balanceada {ba:.3f}")
        if melhor is None or ba > melhor[0]:
            melhor = (ba, clf)
    clf = melhor[1]

    def prob(e):  # garante as 5 colunas mesmo se alguma classe faltou no treino
        p = np.zeros((len(e), len(CLASSES)))
        p[:, clf.classes_] = clf.predict_proba(e)
        return p
    r = relatorio(y["test"], prob(emb["test"]), "test (consistentes)")
    relatorio_ambiguo(ambiguo, prob(emb_amb))
    if emb_ext is not None:
        r["externo"] = relatorio_externo(externo, prob(emb_ext))
    return r


def modo_ajuste(partes, ambiguo, externo, raiz, dev, lote, epocas, saida):
    modelo = timm.create_model(BACKBONE, pretrained=True, num_classes=len(CLASSES)).to(dev)
    tr = DataLoader(Rostos(partes["train"], raiz, transformacoes(modelo, True)),
                    batch_size=lote, shuffle=True, num_workers=WORKERS, drop_last=True)
    contagem = np.bincount(tr.dataset.y, minlength=len(CLASSES)).astype(float)
    peso = torch.tensor((contagem.sum() / np.maximum(contagem, 1)) ** 0.5, dtype=torch.float)
    perda = nn.CrossEntropyLoss(weight=(peso / peso.mean()).to(dev), label_smoothing=0.1)
    # backbone com lr baixa (já sabe rostos), cabeça com lr alta
    cabeca = list(modelo.get_classifier().parameters())
    ids = {id(p) for p in cabeca}
    corpo = [p for p in modelo.parameters() if id(p) not in ids]
    opt = torch.optim.AdamW([{"params": corpo, "lr": 1e-5}, {"params": cabeca, "lr": 1e-3}],
                            weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[1e-5, 1e-3],
                                                total_steps=epocas * len(tr), pct_start=0.1)
    amp = dev == "cuda"
    scaler = torch.amp.GradScaler(enabled=amp)

    @torch.no_grad()
    def prever(df):
        modelo.eval()
        dl = DataLoader(Rostos(df, raiz, transformacoes(modelo, False)), batch_size=lote * 2,
                        num_workers=WORKERS)
        ps = []
        for x, _ in dl:
            with torch.autocast(dev, enabled=amp):
                ps.append(modelo(x.to(dev)).float().softmax(1).cpu().numpy())
        return np.concatenate(ps)

    melhor = -1
    for ep in range(epocas):
        modelo.train()
        t0, soma = time.time(), 0.0
        for x, y in tr:
            x, y = x.to(dev), y.to(dev)
            with torch.autocast(dev, enabled=amp):
                l = perda(modelo(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(l).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            soma += l.item()
        yv = partes["val"].raca.map(CLASSES.index).to_numpy()
        ba = balanced_accuracy_score(yv, prever(partes["val"]).argmax(1))
        print(f"época {ep + 1}/{epocas}: perda {soma / len(tr):.3f}  "
              f"val acurácia balanceada {ba:.3f}  ({time.time() - t0:.0f}s)", flush=True)
        if ba > melhor:
            melhor = ba
            torch.save(modelo.state_dict(), saida / "modelo.pt")

    modelo.load_state_dict(torch.load(saida / "modelo.pt", map_location=dev))
    yt = partes["test"].raca.map(CLASSES.index).to_numpy()
    r = relatorio(yt, prever(partes["test"]), "test (consistentes)")
    relatorio_ambiguo(ambiguo, prever(ambiguo))
    if externo is not None:
        r["externo"] = relatorio_externo(externo, prever(externo))
    return r


def main():
    global CLASSES, BACKBONE, RECORTE, ORIGEM, BINARIO, ALVO
    ap = argparse.ArgumentParser()
    ap.add_argument("--modo", choices=["sonda", "ajuste"], default="sonda")
    ap.add_argument("--dados", type=Path, default=Path(__file__).parent / "dados",
                    help="pasta com manifesto.csv e rostos/")
    ap.add_argument("--saida", type=Path, default=Path(__file__).parent / "resultados")
    ap.add_argument("--por-classe", type=int, help="limite de pessoas por classe no treino")
    ap.add_argument("--epocas", type=int, default=8)
    ap.add_argument("--lote", type=int, default=64)
    ap.add_argument("--semente", type=int, default=0)
    ap.add_argument("--classes", nargs="+", default=["branca", "parda", "preta", "indigena"],
                    help="amarela fica de fora por padrão (rótulo do TSE pouco confiável)")
    ap.add_argument("--externo", type=Path, help="CSV do teste externo (externo.py)")
    ap.add_argument("--recorte-central", type=float, default=1.0,
                    help="usa só a fração central do recorte (ex. 0.62 = sem bordas)")
    ap.add_argument("--backbone", default=BACKBONE,
                    help="na CPU, vit_base_patch32_clip_224.openai é ~4x mais rápido")
    ap.add_argument("--binario", help="ex.: indigena -> modelo indígena vs. outra")
    ap.add_argument("--tag", default="", help="sufixo do arquivo de métricas")
    args = ap.parse_args()

    BACKBONE, RECORTE, ORIGEM, BINARIO = (args.backbone, args.recorte_central,
                                          args.classes, args.binario)
    CLASSES = ["outra", BINARIO] if BINARIO else list(ORIGEM)
    ALVO = BINARIO or ("parda" if "parda" in CLASSES else CLASSES[-1])
    torch.manual_seed(args.semente)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    args.saida.mkdir(parents=True, exist_ok=True)
    partes, ambiguo = carregar(args.dados, args.por_classe, args.semente)
    externo = None
    if args.externo:
        externo = pd.read_csv(args.externo)
        externo = rotular(externo[externo.raca.isin(ORIGEM)])
        if "saturacao" in externo:
            externo = externo[externo.saturacao >= 12]
    print(f"dispositivo: {dev} | backbone: {BACKBONE} | recorte central: {RECORTE} | "
          f"classes: {CLASSES}")
    for s, df in partes.items():
        print(f"{s}: {len(df)} fotos, {df.pessoa_id.nunique()} pessoas, "
              f"{df.raca.value_counts().to_dict()}")
    print(f"ambíguo: {len(ambiguo)} fotos")

    if externo is not None:
        print(f"externo: {len(externo)} fotos, {externo.fonte.value_counts().to_dict()}")

    if args.modo == "sonda":
        r = modo_sonda(partes, ambiguo, externo, args.dados, dev, args.lote)
    else:
        r = modo_ajuste(partes, ambiguo, externo, args.dados, dev, args.lote, args.epocas,
                        args.saida)
    (args.saida / f"metricas_{args.modo}{args.tag}.json").write_text(json.dumps(r, indent=2))


if __name__ == "__main__":
    main()


## 1) Linha de base rápida na GPU (sonda, ~10 min)
Backbone CLIP ViT-B/16 congelado + regressão logística, com TODOS os dados coloridos. Serve para comparar com o fine-tuning.

In [ ]:
%cd /content
!python treino.py --modo sonda --dados /content/dados --externo /content/dados/externo_adorno.csv \
    --classes branca parda preta --lote 256 --saida "$SAIDA/sonda_3classes" --tag _3c 2>&1 | grep -v Warning

## 2) Fine-tuning — branca / parda / preta (~1-2 h na T4)
Salva o melhor modelo (pela validação) a cada época em `resultados/ajuste_3classes/modelo.pt`.

In [ ]:
!python treino.py --modo ajuste --dados /content/dados --externo /content/dados/externo_adorno.csv \
    --classes branca parda preta --epocas 6 --lote 64 --saida "$SAIDA/ajuste_3classes" --tag _3c 2>&1 | grep -v Warning

## 3) Fine-tuning — indígena vs. resto
Modelo separado: no teste anterior, a classe indígena junto das outras "roubava" pardos.

In [ ]:
!python treino.py --modo ajuste --binario indigena --classes branca parda preta indigena \
    --dados /content/dados --externo /content/dados/externo_adorno.csv \
    --epocas 4 --lote 64 --saida "$SAIDA/ajuste_indigena" --tag _ind 2>&1 | grep -v Warning

## 4) Calibração das probabilidades (modelo de 3 classes)
Ajusta temperatura + viés na validação para que "54%" signifique 54%. Mostra o diagrama de confiabilidade e as probabilidades médias por grupo.

In [ ]:
%%writefile /content/calibrar.py
"""Calibra as probabilidades do modelo de 3 classes (branca/parda/preta).

Por que: o fine-tuning usou label smoothing e pesos por classe, que distorcem as
probabilidades. Para dizer "54%" com honestidade, entre as fotos em que o modelo diz ~54%,
~54% precisam ser de fato daquela classe.

Método: ajusta, na VALIDAÇÃO, uma temperatura T (achata/aguça) e um viés por classe b
(corrige a preferência por classe vinda dos pesos): p = softmax(logits / T + b).
Mede no TESTE antes e depois: NLL, Brier, ECE e diagrama de confiabilidade.

Atenção: o treino tem as 3 classes em proporção parecida (amostragem), enquanto a
população brasileira não (Censo 2022: ~45% parda, ~43% branca, ~10% preta). As
probabilidades calibradas valem para essa mistura equilibrada; --prior-censo reajusta.

Uso (Colab):
  python calibrar.py --dados /content/dados --modelo "$SAIDA/ajuste_3classes/modelo.pt" \
      --externo /content/dados/externo_adorno.csv --saida "$SAIDA/calibracao"
"""
import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from torch.utils.data import DataLoader

import treino

CLASSES = ["branca", "parda", "preta"]
CENSO = np.array([0.435, 0.455, 0.102])  # Censo 2022, renormalizado para as 3 classes


@torch.no_grad()
def logits_de(modelo, df, raiz, dev, lote):
    dl = DataLoader(treino.Rostos(df, raiz, treino.transformacoes(modelo, False)),
                    batch_size=lote, num_workers=treino.WORKERS)
    out = []
    for x, _ in dl:
        with torch.autocast(dev, enabled=dev == "cuda"):
            out.append(modelo(x.to(dev)).float().cpu())
    return torch.cat(out)


def ajustar(logits, y):
    """T e b minimizando a NLL na validação."""
    log_t = torch.zeros(1, requires_grad=True)
    b = torch.zeros(logits.shape[1], requires_grad=True)
    opt = torch.optim.LBFGS([log_t, b], lr=0.1, max_iter=500)
    yt = torch.as_tensor(np.asarray(y), dtype=torch.long)

    def passo():
        opt.zero_grad()
        l = torch.nn.functional.cross_entropy(logits / log_t.exp() + b, yt)
        l.backward()
        return l
    opt.step(passo)
    return log_t.exp().item(), b.detach()


def prob(logits, T=1.0, b=None, prior=None):
    z = logits / T + (b if b is not None else 0)
    p = torch.softmax(z, 1).numpy()
    if prior is not None:  # troca o prior equilibrado do treino pelo da população
        treino_prior = np.full(len(CLASSES), 1 / len(CLASSES))
        p = p * prior / treino_prior
        p /= p.sum(1, keepdims=True)
    return p


def metricas(p, y, bins=15):
    n = len(y)
    nll = float(-np.log(np.clip(p[np.arange(n), y], 1e-12, 1)).mean())
    onehot = np.eye(p.shape[1])[y]
    brier = float(((p - onehot) ** 2).sum(1).mean())
    conf, pred = p.max(1), p.argmax(1)
    ece = 0.0
    for lo in np.linspace(0, 1, bins + 1)[:-1]:
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any():
            ece += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return {"nll": round(nll, 4), "brier": round(brier, 4), "ece": round(float(ece), 4)}


def diagrama(p_antes, p_depois, y, caminho):
    fig, eixos = plt.subplots(1, len(CLASSES), figsize=(4 * len(CLASSES), 4), sharey=True)
    bordas = np.linspace(0, 1, 11)
    for k, (ax, c) in enumerate(zip(eixos, CLASSES)):
        for p, nome, cor in [(p_antes, "antes", "#bbbbbb"), (p_depois, "calibrado", "#1f6feb")]:
            idx = np.clip(np.digitize(p[:, k], bordas) - 1, 0, 9)
            xs, ys = [], []
            for i in range(10):
                m = idx == i
                if m.sum() >= 20:
                    xs.append(p[m, k].mean())
                    ys.append((y[m] == k).mean())
            ax.plot(xs, ys, "o-", color=cor, label=nome)
        ax.plot([0, 1], [0, 1], "--", color="#888", lw=1)
        ax.set_title(f"P({c})")
        ax.set_xlabel("probabilidade dita pelo modelo")
    eixos[0].set_ylabel("frequência real no teste")
    eixos[0].legend()
    fig.suptitle("Diagrama de confiabilidade (perfeito = diagonal)")
    fig.tight_layout()
    fig.savefig(caminho, dpi=120)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dados", type=Path, required=True)
    ap.add_argument("--modelo", type=Path, help="modelo.pt do fine-tuning (sem = só teste)")
    ap.add_argument("--backbone", default="vit_base_patch16_clip_224.openai")
    ap.add_argument("--externo", type=Path)
    ap.add_argument("--saida", type=Path, required=True)
    ap.add_argument("--lote", type=int, default=128)
    ap.add_argument("--max-ambiguo", type=int, default=8000)
    ap.add_argument("--por-classe", type=int, help="só para teste rápido")
    args = ap.parse_args()

    treino.CLASSES = treino.ORIGEM = CLASSES
    treino.BACKBONE, treino.BINARIO, treino.ALVO = args.backbone, None, "parda"
    args.saida.mkdir(parents=True, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    partes, amb = treino.carregar(args.dados, args.por_classe, 0)
    amb = amb.sample(min(len(amb), args.max_ambiguo), random_state=0)
    modelo = timm.create_model(args.backbone, pretrained=args.modelo is None,
                               num_classes=len(CLASSES))
    if args.modelo:
        modelo.load_state_dict(torch.load(args.modelo, map_location="cpu"))
    modelo = modelo.to(dev).eval()

    print("calculando logits...", flush=True)
    L = {s: logits_de(modelo, partes[s], args.dados, dev, args.lote) for s in ["val", "test"]}
    y = {s: partes[s].raca.map(CLASSES.index).to_numpy() for s in ["val", "test"]}
    T, b = ajustar(L["val"], y["val"])
    print(f"\ntemperatura T = {T:.3f}   viés por classe b = "
          f"{dict(zip(CLASSES, b.numpy().round(3)))}")

    p0, p1 = prob(L["test"]), prob(L["test"], T, b)
    r = {"T": T, "b": b.tolist(), "teste_antes": metricas(p0, y["test"]),
         "teste_calibrado": metricas(p1, y["test"])}
    print("\n=== teste: antes vs. calibrado (menor = melhor) ===")
    print(pd.DataFrame({"antes": r["teste_antes"], "calibrado": r["teste_calibrado"]}).T)
    diagrama(p0, p1, y["test"], args.saida / "confiabilidade.png")

    # média das probabilidades calibradas por grupo: os ambíguos devem ficar "no meio"
    def resumo(df, p, grupo):
        t = pd.DataFrame(p, columns=CLASSES).assign(g=df[grupo].values)
        t["nota_0a2"] = t.parda + 2 * t.preta   # 0 = branca, 1 = parda, 2 = preta
        return (t.groupby("g").mean() * [100, 100, 100, 1]).round(
            {"branca": 1, "parda": 1, "preta": 1, "nota_0a2": 2})

    print("\n=== média das probabilidades calibradas (%), teste consistente ===")
    print(resumo(partes["test"], p1, "raca").to_string())
    pa = prob(logits_de(modelo, amb, args.dados, dev, args.lote), T, b)
    top = amb.racas_historico.value_counts().head(6).index
    m = amb.racas_historico.isin(top).values
    print("\n=== idem, pessoas que mudaram de declaração ===")
    print(resumo(amb[m], pa[m], "racas_historico").to_string())

    if args.externo:
        ext = pd.read_csv(args.externo)
        ext = ext[ext.raca.isin(CLASSES) & (ext.saturacao >= 12)]
        pe = prob(logits_de(modelo, ext, args.dados, dev, args.lote), T, b)
        print("\n=== idem, teste externo ===")
        print(resumo(ext, pe, "fonte").to_string())

    print("\n=== com prior do Censo 2022 (população real, não a amostra equilibrada) ===")
    pc = prob(L["test"], T, b, prior=CENSO)
    print(resumo(partes["test"], pc, "raca").to_string())

    # distribuição: quão "decididas" são as respostas para quem sempre se declarou parda
    pp = p1[y["test"] == 1, 1]
    print(f"\nP(parda) entre pardos consistentes: mediana {np.median(pp):.0%}, "
          f"{(pp >= 0.7).mean():.0%} acima de 70%, {(pp < 0.3).mean():.0%} abaixo de 30%")

    (args.saida / "calibracao.json").write_text(json.dumps(r, indent=2))
    print(f"\nsalvo em {args.saida} (calibracao.json, confiabilidade.png)")


if __name__ == "__main__":
    main()


In [ ]:
%cd /content
!python calibrar.py --dados /content/dados --modelo "$SAIDA/ajuste_3classes/modelo.pt" \
    --externo /content/dados/externo_adorno.csv --saida "$SAIDA/calibracao" 2>&1 | grep -v Warning
from IPython.display import Image
Image(f"{SAIDA}/calibracao/confiabilidade.png")

## 5) Destilação: modelo pequeno para rodar no navegador
O professor (B/16) rotula todas as fotos com probabilidades; um MobileNetV3 (~4 M parâmetros, 160×160) aprende a imitá-las. Exporta `aluno/modelo.onnx` + `meta.json`.

In [ ]:
%%writefile /content/destilar.py
"""Destila o modelo grande (CLIP ViT-B/16, "professor") num modelo pequeno ("aluno") que
roda no navegador do celular.

1. O professor calcula as probabilidades (calibradas) de todas as fotos coloridas:
   consistentes E quem mudou de declaração. Para estes, a probabilidade do professor é
   justamente o "meio-termo", então eles entram no treino do aluno (só os do split train).
2. O aluno (MobileNetV3, ~5 M parâmetros, entrada 160x160 = o próprio recorte, sem resize)
   aprende a imitar essas probabilidades (entropia cruzada com alvo suave).
3. Avalia aluno vs. professor no teste e exporta modelo.onnx + meta.json.

Uso (Colab):
  python destilar.py --dados /content/dados --professor "$SAIDA/ajuste_3classes/modelo.pt" \
      --calibracao "$SAIDA/calibracao/calibracao.json" --saida "$SAIDA/aluno" \
      --externo /content/dados/externo_adorno.csv
"""
import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import balanced_accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

import treino

CLASSES = ["branca", "parda", "preta"]
LADO = 160
MEDIA, DESVIO = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ImageNet (timm mobilenet)


class Fotos(Dataset):
    def __init__(self, arquivos, raiz, alvo, transform):
        self.arquivos = [raiz / a for a in arquivos]
        self.alvo = torch.as_tensor(np.asarray(alvo), dtype=torch.float32)
        self.t = transform

    def __len__(self):
        return len(self.arquivos)

    def __getitem__(self, i):
        return self.t(Image.open(self.arquivos[i]).convert("RGB")), self.alvo[i]


def transf_aluno(treinando: bool):
    norm = [T.ToTensor(), T.Normalize(MEDIA, DESVIO)]
    if not treinando:  # no navegador: recorte 160x160 -> RGB/255 -> normaliza. Nada mais.
        return T.Compose([T.Resize((LADO, LADO))] + norm)
    return T.Compose([T.RandomResizedCrop(LADO, scale=(0.6, 1.0), ratio=(0.9, 1.1)),
                      T.RandomHorizontalFlip()] + norm)


@torch.no_grad()
def prever(modelo, ds, dev, lote):
    modelo.eval()
    out = []
    for x, _ in DataLoader(ds, batch_size=lote, num_workers=treino.WORKERS):
        with torch.autocast(dev, enabled=dev == "cuda"):
            out.append(modelo(x.to(dev)).float().cpu())
    return torch.cat(out)


def logits_professor(df, args, dev, cache: Path):
    """Logits do professor para todas as fotos (com cache: leva ~20 min na T4)."""
    if cache.exists():
        z = np.load(cache, allow_pickle=True)
        mapa = dict(zip(z["arquivos"], z["logits"]))
        if all(a in mapa for a in df.arquivo):
            print("logits do professor: cache", flush=True)
            return torch.as_tensor(np.stack([mapa[a] for a in df.arquivo]))
    prof = timm.create_model(args.backbone, pretrained=args.professor is None,
                             num_classes=len(CLASSES))
    if args.professor:
        prof.load_state_dict(torch.load(args.professor, map_location="cpu"))
    prof = prof.to(dev)
    ds = treino.Rostos(df.assign(raca=CLASSES[0]), args.dados, treino.transformacoes(prof, False))
    t0 = time.time()
    print(f"professor: calculando {len(df)} fotos...", flush=True)
    L = prever(prof, ds, dev, args.lote * 2)
    print(f"  {time.time() - t0:.0f}s", flush=True)
    cache.parent.mkdir(parents=True, exist_ok=True)
    np.savez(cache, arquivos=df.arquivo.to_numpy(), logits=L.numpy())
    del prof
    if dev == "cuda":
        torch.cuda.empty_cache()
    return L


def ece(p, y, bins=15):
    conf, pred = p.max(1), p.argmax(1)
    e = 0.0
    for lo in np.linspace(0, 1, bins + 1)[:-1]:
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)


def resumo(p, y, nome):
    pred = p.argmax(1)
    ip = CLASSES.index("parda")
    r = {"acuracia_balanceada": round(balanced_accuracy_score(y, pred), 4),
         "recall_parda": round(float((pred[y == ip] == ip).mean()), 4),
         "precisao_parda": round(float((y[pred == ip] == ip).mean()), 4),
         "ece": round(ece(p, y), 4),
         "nll": round(float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()), 4)}
    print(f"{nome:10s} " + "  ".join(f"{k} {v}" for k, v in r.items()))
    return r


def notas(df, p, grupo):
    t = pd.DataFrame(p * 100, columns=CLASSES).assign(g=df[grupo].values)
    t["nota_0a2"] = (t.parda + 2 * t.preta) / 100
    return t.groupby("g").mean().round(2)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dados", type=Path, required=True)
    ap.add_argument("--professor", type=Path, help="modelo.pt do fine-tuning")
    ap.add_argument("--backbone", default="vit_base_patch16_clip_224.openai")
    ap.add_argument("--calibracao", type=Path, help="calibracao.json (T e b do professor)")
    ap.add_argument("--aluno", default="mobilenetv3_large_100")
    ap.add_argument("--saida", type=Path, required=True)
    ap.add_argument("--externo", type=Path)
    ap.add_argument("--epocas", type=int, default=12)
    ap.add_argument("--lote", type=int, default=128)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--limite", type=int, help="só para teste rápido: N fotos por conjunto")
    args = ap.parse_args()

    treino.CLASSES = treino.ORIGEM = CLASSES
    treino.BACKBONE, treino.BINARIO, treino.ALVO = args.backbone, None, "parda"
    args.saida.mkdir(parents=True, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)

    partes, amb = treino.carregar(args.dados, None, 0)
    cons = pd.concat([partes[s] for s in ["train", "val", "test"]])
    todas = pd.concat([cons.assign(grupo="consistente"), amb.assign(grupo="ambiguo")])
    if args.limite:
        todas = todas.sample(frac=1, random_state=0)
        todas = todas[todas.groupby(["split", "grupo"]).cumcount() < args.limite]
    todas = todas.reset_index(drop=True)
    ext = None
    if args.externo:
        ext = pd.read_csv(args.externo)
        ext = ext[ext.raca.isin(CLASSES) & (ext.saturacao >= 12)].reset_index(drop=True)
    print(todas.groupby(["grupo", "split"]).size().to_string(), flush=True)

    # 1) professor
    L = logits_professor(todas, args, dev, args.saida / "logits_professor.npz")
    Tp, bp = 1.0, torch.zeros(len(CLASSES))
    if args.calibracao and args.calibracao.exists():
        c = json.loads(args.calibracao.read_text())
        Tp, bp = c["T"], torch.tensor(c["b"])
    P = torch.softmax(L / Tp + bp, 1).numpy()

    # 2) aluno
    tr = todas.split.eq("train").to_numpy()
    va = todas.split.eq("val").to_numpy() & todas.grupo.eq("consistente").to_numpy()
    aluno = timm.create_model(args.aluno, pretrained=True, num_classes=len(CLASSES))
    for w in aluno.get_classifier().parameters():  # começa em 33/33/33, não em logits ~5
        torch.nn.init.zeros_(w)
    aluno = aluno.to(dev)
    print(f"aluno {args.aluno}: {sum(p.numel() for p in aluno.parameters()) / 1e6:.1f} M "
          f"parâmetros | treino {tr.sum()} fotos (alvo suave do professor)", flush=True)
    dl = DataLoader(Fotos(todas.arquivo[tr], args.dados, P[tr], transf_aluno(True)),
                    batch_size=args.lote, shuffle=True, num_workers=treino.WORKERS,
                    drop_last=True)
    ds_val = Fotos(todas.arquivo[va], args.dados, P[va], transf_aluno(False))
    y_val = todas.raca[va].map(CLASSES.index).to_numpy()
    opt = torch.optim.AdamW(aluno.parameters(), lr=args.lr, weight_decay=0.02)
    passos = args.epocas * len(dl)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=args.lr, total_steps=passos,
                                                pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=dev == "cuda")
    melhor = math.inf
    for ep in range(args.epocas):
        aluno.train()
        t0, soma = time.time(), 0.0
        for x, alvo in dl:
            x, alvo = x.to(dev), alvo.to(dev)
            with torch.autocast(dev, enabled=dev == "cuda"):
                saida = aluno(x)
            perda = -(alvo * F.log_softmax(saida.float(), 1)).sum(1).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(perda).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            soma += perda.item()
        pv = torch.softmax(prever(aluno, ds_val, dev, args.lote * 2), 1).numpy()
        # KL até o professor: o quanto o aluno ainda se afasta das probabilidades dele
        kl = float((P[va] * (np.log(np.clip(P[va], 1e-9, 1)) - np.log(np.clip(pv, 1e-9, 1))))
                   .sum(1).mean())
        ba = balanced_accuracy_score(y_val, pv.argmax(1))
        print(f"época {ep + 1}/{args.epocas}: perda {soma / len(dl):.4f}  val KL {kl:.4f}  "
              f"val acurácia balanceada {ba:.3f}  ({time.time() - t0:.0f}s)", flush=True)
        if kl < melhor:
            melhor = kl
            torch.save(aluno.state_dict(), args.saida / "aluno.pt")

    # 3) avaliação: aluno vs. professor
    aluno.load_state_dict(torch.load(args.saida / "aluno.pt", map_location=dev))
    te = todas.split.eq("test").to_numpy()
    ds_te = Fotos(todas.arquivo[te], args.dados, P[te], transf_aluno(False))
    pa = torch.softmax(prever(aluno, ds_te, dev, args.lote * 2), 1).numpy()
    tc = todas.grupo[te].eq("consistente").to_numpy()
    y_te = todas.raca[te][tc].map(CLASSES.index).to_numpy()
    print("\n=== teste (consistentes) ===")
    r = {"professor": resumo(P[te][tc], y_te, "professor"),
         "aluno": resumo(pa[tc], y_te, "aluno")}
    concord = float((pa.argmax(1) == P[te].argmax(1)).mean())
    dif = float(np.abs(pa - P[te]).max(1).mean())
    print(f"aluno concorda com o professor em {concord:.1%} das fotos de teste; "
          f"diferença média da maior probabilidade: {dif * 100:.1f} pontos")
    r["concordancia"], r["dif_media_pontos"] = concord, dif * 100

    dte = todas[te].reset_index(drop=True)
    g = np.where(dte.grupo == "consistente", "sempre " + dte.raca, dte.racas_historico)
    dte = dte.assign(g=g)
    top = pd.Series(g).value_counts().head(8).index
    m = dte.g.isin(top).to_numpy()
    print("\n=== nota 0-2 por grupo (teste): professor | aluno ===")
    print(pd.concat({"professor": notas(dte[m], P[te][m], "g").nota_0a2,
                     "aluno": notas(dte[m], pa[m], "g").nota_0a2}, axis=1)
          .sort_values("professor").to_string())

    if ext is not None:
        pe = torch.softmax(prever(aluno, Fotos(ext.arquivo, args.dados, np.zeros((len(ext), 3)),
                                               transf_aluno(False)), dev, args.lote * 2), 1)
        print("\n=== teste externo (aluno) ===")
        print(notas(ext, pe.numpy(), "fonte").to_string())

    # 4) exporta para o navegador
    aluno = aluno.float().cpu().eval()
    onnx = args.saida / "modelo.onnx"
    torch.onnx.export(aluno, torch.randn(1, 3, LADO, LADO), str(onnx), opset_version=17,
                      input_names=["imagem"], output_names=["logits"],
                      dynamic_axes={"imagem": {0: "n"}, "logits": {0: "n"}}, dynamo=False)
    meta = {"classes": CLASSES, "lado": LADO, "media": MEDIA, "desvio": DESVIO,
            "entrada": "RGB 0-1, NCHW, recorte alinhado pelos olhos (OLHOS_Y 0.40, "
                       "DIST_OLHOS 0.28, borda replicada)",
            "aluno": args.aluno, "metricas": r}
    (args.saida / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    print(f"\nexportado: {onnx} ({onnx.stat().st_size / 1e6:.1f} MB) + meta.json")


if __name__ == "__main__":
    main()


In [ ]:
!pip -q install onnx
%cd /content
!python destilar.py --dados /content/dados --professor "$SAIDA/ajuste_3classes/modelo.pt" \
    --calibracao "$SAIDA/calibracao/calibracao.json" --externo /content/dados/externo_adorno.csv \
    --saida "$SAIDA/aluno" 2>&1 | grep -v -i warn

## 6) Aluno com duas saídas (branca/parda/preta + indígena)
Um só modelo pequeno imita os dois professores. Exporta `aluno2/modelo.onnx` + `meta.json`.

In [ ]:
%%writefile /content/destilar2.py
"""Destila DOIS professores num único aluno pequeno com duas saídas:

  saída A (3 logits): branca / parda / preta    <- professor ajuste_3classes (calibrado)
  saída B (2 logits): outra / indígena          <- professor ajuste_indigena (calibrado aqui)

O aluno vê todas as fotos coloridas (inclusive indígenas e quem mudou de declaração) e
aprende as probabilidades dos dois professores. Mesmo tamanho do aluno anterior.

Uso (Colab):
  python destilar2.py --dados /content/dados --saida "$SAIDA/aluno2" \
      --prof3 "$SAIDA/ajuste_3classes/modelo.pt" --cal3 "$SAIDA/calibracao/calibracao.json" \
      --profi "$SAIDA/ajuste_indigena/modelo.pt" --externo /content/dados/externo_adorno.csv \
      --cache3 "$SAIDA/aluno/logits_professor.npz"
"""
import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from torch.utils.data import DataLoader, WeightedRandomSampler

import destilar
import treino
from destilar import Fotos, LADO, MEDIA, DESVIO, transf_aluno, prever, ece

TRES = ["branca", "parda", "preta"]
QUATRO = TRES + ["indigena"]


def logits_de(caminho_pt, n_saidas, df, args, dev, cache: Path, cache_extra: Path | None = None):
    """Logits de um professor para todas as fotos de df, reaproveitando caches por arquivo."""
    mapa = {}
    for c in [cache_extra, cache]:
        if c and c.exists():
            z = np.load(c, allow_pickle=True)
            mapa.update(zip(z["arquivos"], z["logits"]))
    faltam = df[~df.arquivo.isin(mapa)].drop_duplicates("arquivo")
    if len(faltam):
        prof = timm.create_model(args.backbone, pretrained=False, num_classes=n_saidas)
        prof.load_state_dict(torch.load(caminho_pt, map_location="cpu"))
        prof = prof.to(dev)
        print(f"  {caminho_pt.parent.name}: {len(faltam)} fotos ({len(df) - len(faltam)} em cache)",
              flush=True)
        t0 = time.time()
        L = prever(prof, treino.Rostos(faltam.assign(raca="branca"), args.dados,
                                       treino.transformacoes(prof, False)), dev, args.lote * 2)
        print(f"  {time.time() - t0:.0f}s", flush=True)
        mapa.update(zip(faltam.arquivo, L.numpy()))
        np.savez(cache, arquivos=np.array(list(mapa)), logits=np.stack(list(mapa.values())))
        del prof
        if dev == "cuda":
            torch.cuda.empty_cache()
    return torch.as_tensor(np.stack([mapa[a] for a in df.arquivo]))


def calibrar(logits, y):
    log_t = torch.zeros(1, requires_grad=True)
    b = torch.zeros(logits.shape[1], requires_grad=True)
    opt = torch.optim.LBFGS([log_t, b], lr=0.1, max_iter=500)
    yt = torch.as_tensor(y, dtype=torch.long)

    def passo():
        opt.zero_grad()
        l = F.cross_entropy(logits / log_t.exp() + b, yt)
        l.backward()
        return l
    opt.step(passo)
    return log_t.exp().item(), b.detach()


def ece_bin(p, y, bins=15):
    """ECE da probabilidade de indígena (não da classe prevista): o que o site vai mostrar."""
    e = 0.0
    for lo in np.linspace(0, 1, bins + 1)[:-1]:
        m = (p > lo) & (p <= lo + 1 / bins)
        if m.any():
            e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dados", type=Path, required=True)
    ap.add_argument("--saida", type=Path, required=True)
    ap.add_argument("--prof3", type=Path, required=True)
    ap.add_argument("--cal3", type=Path)
    ap.add_argument("--profi", type=Path, required=True)
    ap.add_argument("--cache3", type=Path, help="logits do professor 3 classes já calculados")
    ap.add_argument("--backbone", default="vit_base_patch16_clip_224.openai")
    ap.add_argument("--aluno", default="mobilenetv3_large_100")
    ap.add_argument("--externo", type=Path)
    ap.add_argument("--epocas", type=int, default=12)
    ap.add_argument("--lote", type=int, default=128)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--peso-indigena", type=float, default=4.0,
                    help="sobreamostragem de fotos de indígenas no treino do aluno")
    ap.add_argument("--limite", type=int, help="só para teste rápido")
    args = ap.parse_args()

    treino.BACKBONE = args.backbone
    args.saida.mkdir(parents=True, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)

    m = pd.read_csv(args.dados / "manifesto.csv", dtype={"sq": str, "pessoa_id": str})
    m = m[(m.status == "ok") & (m.saturacao >= 12) & m.raca.isin(QUATRO)]
    m = m[m.racas_historico.map(lambda h: all(r in QUATRO for r in h.split("|")))]
    hist = m.racas_historico.str.split("|")
    m = m.assign(
        tri_cons=(m.n_racas == 1) & m.raca.isin(TRES),             # avalia a saída A
        ind_y=hist.map(lambda h: 1 if h == ["indigena"] else (0 if "indigena" not in h else -1)),
    ).reset_index(drop=True)                                       # ind_y -1 = ambíguo
    if args.limite:
        m = m.sample(frac=1, random_state=0)
        m = m[m.groupby(["split", "ind_y"]).cumcount() < args.limite].reset_index(drop=True)
    print(f"{len(m)} fotos | indígenas consistentes {(m.ind_y == 1).sum()} | "
          f"ambíguos indígena/outra {(m.ind_y == -1).sum()}", flush=True)

    # 1) professores
    print("professores:", flush=True)
    L3 = logits_de(args.prof3, 3, m, args, dev, args.saida / "logits_prof3.npz", args.cache3)
    Li = logits_de(args.profi, 2, m, args, dev, args.saida / "logits_profi.npz")
    T3, b3 = 1.0, torch.zeros(3)
    if args.cal3 and args.cal3.exists():
        c = json.loads(args.cal3.read_text())
        T3, b3 = c["T"], torch.tensor(c["b"])
    P3 = torch.softmax(L3 / T3 + b3, 1).numpy()

    val = (m.split == "val").to_numpy() & (m.ind_y >= 0).to_numpy()
    Ti, bi = calibrar(Li[val], m.ind_y[val].to_numpy())
    Pi = torch.softmax(Li / Ti + bi, 1).numpy()
    te_i = (m.split == "test").to_numpy() & (m.ind_y >= 0).to_numpy()
    yi = m.ind_y[te_i].to_numpy()
    p_cru = torch.softmax(Li[te_i], 1).numpy()[:, 1]
    print(f"\nprofessor indígena: T={Ti:.3f} b={bi.numpy().round(3)} | ECE de P(indígena) no teste: "
          f"antes {ece_bin(p_cru, yi):.1%}, calibrado {ece_bin(Pi[te_i, 1], yi):.1%} | "
          f"AUC {roc_auc_score(yi, Pi[te_i, 1]):.3f}", flush=True)

    # 2) aluno com duas saídas
    alvo = np.concatenate([P3, Pi], 1)              # 5 colunas
    tr = (m.split == "train").to_numpy()
    aluno = timm.create_model(args.aluno, pretrained=True, num_classes=5)
    for w in aluno.get_classifier().parameters():
        torch.nn.init.zeros_(w)
    aluno = aluno.to(dev)
    ds_tr = Fotos(m.arquivo[tr], args.dados, alvo[tr], transf_aluno(True))
    pesos = np.where(m.ind_y[tr] != 0, args.peso_indigena, 1.0)
    # sobreamostrar não distorce o alvo: cada foto tem a própria probabilidade a imitar
    dl = DataLoader(ds_tr, batch_size=args.lote, num_workers=treino.WORKERS, drop_last=True,
                    sampler=WeightedRandomSampler(pesos, num_samples=len(pesos), replacement=True))
    va = (m.split == "val").to_numpy()
    ds_va = Fotos(m.arquivo[va], args.dados, alvo[va], transf_aluno(False))
    print(f"\naluno {args.aluno}: treino {tr.sum()} fotos, indígenas sobreamostrados x"
          f"{args.peso_indigena:g}", flush=True)
    opt = torch.optim.AdamW(aluno.parameters(), lr=args.lr, weight_decay=0.02)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=args.lr,
                                                total_steps=args.epocas * len(dl), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=dev == "cuda")

    def perda_de(z, a):
        la = -(a[:, :3] * F.log_softmax(z[:, :3], 1)).sum(1).mean()
        lb = -(a[:, 3:] * F.log_softmax(z[:, 3:], 1)).sum(1).mean()
        return la + lb

    def kl(p, q):
        return float((p * (np.log(np.clip(p, 1e-9, 1)) - np.log(np.clip(q, 1e-9, 1)))).sum(1).mean())

    melhor = math.inf
    for ep in range(args.epocas):
        aluno.train()
        t0, soma = time.time(), 0.0
        for x, a in dl:
            x, a = x.to(dev), a.to(dev)
            with torch.autocast(dev, enabled=dev == "cuda"):
                z = aluno(x)
            perda = perda_de(z.float(), a)
            opt.zero_grad(set_to_none=True)
            scaler.scale(perda).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            soma += perda.item()
        z = prever(aluno, ds_va, dev, args.lote * 2)
        k3 = kl(alvo[va, :3], torch.softmax(z[:, :3], 1).numpy())
        ki = kl(alvo[va, 3:], torch.softmax(z[:, 3:], 1).numpy())
        print(f"época {ep + 1}/{args.epocas}: perda {soma / len(dl):.4f}  val KL 3 classes {k3:.4f}"
              f"  val KL indígena {ki:.4f}  ({time.time() - t0:.0f}s)", flush=True)
        if k3 + ki < melhor:
            melhor = k3 + ki
            torch.save(aluno.state_dict(), args.saida / "aluno.pt")

    # 3) avaliação
    aluno.load_state_dict(torch.load(args.saida / "aluno.pt", map_location=dev))
    te = (m.split == "test").to_numpy()
    z = prever(aluno, Fotos(m.arquivo[te], args.dados, alvo[te], transf_aluno(False)), dev,
               args.lote * 2)
    pa3, pai = torch.softmax(z[:, :3], 1).numpy(), torch.softmax(z[:, 3:], 1).numpy()[:, 1]
    mt = m[te].reset_index(drop=True)
    tc = mt.tri_cons.to_numpy()
    y3 = mt.raca[tc].map(TRES.index).to_numpy()
    r = {}
    print("\n=== saída A: branca/parda/preta (teste, consistentes) ===")
    r["prof3"] = destilar.resumo(P3[te][tc], y3, "professor")
    r["aluno3"] = destilar.resumo(pa3[tc], y3, "aluno")

    ic = (mt.ind_y >= 0).to_numpy()
    yi = mt.ind_y[ic].to_numpy()
    print("\n=== saída B: indígena (teste) ===")
    for nome, p in [("professor", Pi[te, 1][ic]), ("aluno", pai[ic])]:
        pred = p >= 0.5
        r[f"ind_{nome}"] = {"auc": round(roc_auc_score(yi, p), 4),
                            "recall": round(float(pred[yi == 1].mean()), 4),
                            "precisao": round(float(yi[pred].mean()) if pred.any() else 0, 4),
                            "ece_p_indigena": round(ece_bin(p, yi), 4)}
        print(f"{nome:10s} " + "  ".join(f"{k} {v}" for k, v in r[f"ind_{nome}"].items()))
    print("P(indígena) média por grupo (aluno):")
    g = np.where(mt.ind_y == 1, "indígena consistente",
                 np.where(mt.ind_y == -1, "mudou indígena<->outra", "sempre " + mt.raca))
    print((pd.Series(pai).groupby(g).mean() * 100).round(1).to_string())

    if args.externo:
        ext = pd.read_csv(args.externo)
        ext = ext[ext.raca.isin(QUATRO) & (ext.saturacao >= 12)].reset_index(drop=True)
        ze = prever(aluno, Fotos(ext.arquivo, args.dados, np.zeros((len(ext), 5)),
                                 transf_aluno(False)), dev, args.lote * 2)
        pe3, pei = torch.softmax(ze[:, :3], 1).numpy(), torch.softmax(ze[:, 3:], 1).numpy()[:, 1]
        t = pd.DataFrame(pe3 * 100, columns=TRES).assign(indigena=pei * 100, fonte=ext.fonte)
        print("\n=== teste externo (aluno, médias %) ===")
        print(t.groupby("fonte").mean().round(1).to_string())
        print("fração com P(indígena) >= 50%:")
        print((t.assign(i50=t.indigena >= 50).groupby("fonte").i50.mean() * 100).round(1).to_string())

    # 4) exporta
    aluno = aluno.float().cpu().eval()
    onnx = args.saida / "modelo.onnx"
    torch.onnx.export(aluno, torch.randn(1, 3, LADO, LADO), str(onnx), opset_version=17,
                      input_names=["imagem"], output_names=["logits"],
                      dynamic_axes={"imagem": {0: "n"}, "logits": {0: "n"}}, dynamo=False)
    meta = {"saidas": {"A": {"logits": [0, 3], "classes": TRES},
                       "B": {"logits": [3, 5], "classes": ["outra", "indigena"]}},
            "lado": LADO, "media": MEDIA, "desvio": DESVIO, "aluno": args.aluno,
            "calibracao_indigena": {"T": Ti, "b": bi.tolist()}, "metricas": r}
    (args.saida / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    print(f"\nexportado: {onnx} ({onnx.stat().st_size / 1e6:.1f} MB) + meta.json")


if __name__ == "__main__":
    main()


In [ ]:
!pip -q install onnx
%cd /content
!python destilar2.py --dados /content/dados --saida "$SAIDA/aluno2" \
    --prof3 "$SAIDA/ajuste_3classes/modelo.pt" --cal3 "$SAIDA/calibracao/calibracao.json" \
    --profi "$SAIDA/ajuste_indigena/modelo.pt" --cache3 "$SAIDA/aluno/logits_professor.npz" \
    --externo /content/dados/externo_adorno.csv 2>&1 | grep -v -i warn